> **Versión estudiante.** Complete las celdas marcadas con `# TODO`. La solución completa está disponible en `notebooks-soluciones/sesion-06-redes-neuronales.ipynb`.


# Sesión 6 · Redes neuronales para predecir la demanda
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

> Ejecute cada celda con `Shift + Enter` y lea la explicación antes de cada paso. Todos los datos son **sintéticos** (simulados) para practicar.

**Objetivo:** entrenar una red neuronal pequeña, vigilar su curva de pérdida y compararla con una regresión lineal.

> Primero use [TensorFlow Playground](https://playground.tensorflow.org) para ver una red aprendiendo.

## 1. Demanda horaria simulada
La demanda depende de la hora del día, del día de la semana y de la temperatura.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rng = np.random.default_rng(3)
t = pd.date_range('2025-01-01', periods=24*150, freq='h')
hora = t.hour.values; dsem = t.dayofweek.values
temp = 20 + 5*np.sin((hora-9)/24*2*np.pi) + rng.normal(0,1.5,len(t))
dem = 300 + 120*np.exp(-((hora-19)/3.5)**2) + 80*np.exp(-((hora-8)/2.5)**2) - 40*(dsem>=5) + 6*(temp-20)**2/5 + rng.normal(0,10,len(t))
df = pd.DataFrame({'hora':hora,'dsem':dsem,'temp':temp,'dem':dem}, index=t)
df['sin_h']=np.sin(2*np.pi*hora/24); df['cos_h']=np.cos(2*np.pi*hora/24)
df.head()

## 2. Variables, división y escalado
Usamos la hora como seno y coseno para que 23 h y 0 h queden cerca. Dividimos por fecha y escalamos con train.

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ['sin_h','cos_h','dsem','temp']
corte = int(len(df)*0.8)
Xtr, Xte = df[cols].iloc[:corte], df[cols].iloc[corte:]
ytr, yte = df['dem'].iloc[:corte], df['dem'].iloc[corte:]
ex = StandardScaler().fit(Xtr); ey = StandardScaler().fit(ytr.values.reshape(-1,1))
Xtr_s, Xte_s = ex.transform(Xtr), ex.transform(Xte)
ytr_s = ey.transform(ytr.values.reshape(-1,1)).ravel()

## 3. Línea base: regresión lineal

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
lin = LinearRegression().fit(Xtr_s, ytr_s)
p_lin = ey.inverse_transform(lin.predict(Xte_s).reshape(-1,1)).ravel()
print('MAE regresión lineal:', round(mean_absolute_error(yte, p_lin),2))

## 4. Red neuronal con scikit-learn
Dos capas ocultas (32 y 16 neuronas) con activación ReLU. `early_stopping` detiene el entrenamiento si la validación no mejora.

In [ ]:
# TODO: Cree y entrene la red neuronal con MLPRegressor.
# Pistas:
# - Use 2 capas ocultas, por ejemplo hidden_layer_sizes=(32,16)
# - activation='relu'
# - max_iter=300, early_stopping=True, random_state=0
# - Entrene con .fit(Xtr_s, ytr_s)
# - Calcule las predicciones con .predict(Xte_s) y des-escale con ey.inverse_transform(...)
# - Imprima el MAE con mean_absolute_error(yte, p_red)
from sklearn.neural_network import MLPRegressor
red = None  # TODO: red = MLPRegressor(hidden_layer_sizes=..., activation=..., max_iter=..., early_stopping=..., random_state=...)
# TODO: red.fit(Xtr_s, ytr_s)
p_red = None  # TODO: p_red = ey.inverse_transform(red.predict(Xte_s).reshape(-1,1)).ravel()
# TODO: print('MAE red neuronal:', round(mean_absolute_error(yte, p_red),2))


## 5. Curva de pérdida
La pérdida debe bajar. Si en validación sube, hay sobreajuste.

In [ ]:
# TODO: Grafique la curva de pérdida de entrenamiento y el puntaje de validación.
# Pistas:
# - red.loss_curve_ contiene la pérdida de entrenamiento por época
# - red.validation_scores_ contiene el R² de validación por época
# - Use plt.plot(...) para cada una (con label='entrenamiento' y label='puntaje validación (R²)')
# - No olvide plt.xlabel('época'), plt.legend() y plt.show()


## 6. Real contra predicho en una semana

In [ ]:
# TODO: Compare las predicciones (red y lineal) con los valores reales de una semana del set de prueba.
# Pistas:
# - n = 24*7 (una semana de datos horarios)
# - Grafique yte.values[:n], p_red[:n] y p_lin[:n] en la misma figura, cada uno con su label
# - Use plt.figure(figsize=(10,3.5)), plt.legend() y plt.show()


## 7. Versión Keras (opcional)
En Google Colab, TensorFlow y Keras ya vienen instalados. Esta celda hace lo mismo con Keras; si su entorno no lo tiene, el código lo avisa.

In [ ]:
try:
    import tensorflow as tf
    tf.random.set_seed(0)
    # TODO: Defina el modelo Keras Sequential.
    # Pistas:
    # - tf.keras.layers.Input(shape=(Xtr_s.shape[1],))
    # - Dos capas ocultas Dense con activation='relu': 32 neuronas y 16 neuronas
    # - Capa de salida: Dense(1) sin activación (es una regresión)
    modelo = None  # TODO: modelo = tf.keras.Sequential([...])
    # TODO: Compile el modelo: modelo.compile(optimizer='adam', loss='mse')
    # TODO: Entrene: hist = modelo.fit(Xtr_s, ytr_s, epochs=50, validation_split=0.2, verbose=0)
    hist = None
    p_k = None  # TODO: p_k = ey.inverse_transform(modelo.predict(Xte_s, verbose=0)).ravel()
    print('MAE Keras:', round(mean_absolute_error(yte, p_k),2))
    plt.plot(hist.history['loss'], label='train'); plt.plot(hist.history['val_loss'], label='val'); plt.legend(); plt.show()
except ImportError:
    print('TensorFlow no está instalado en este entorno; use la red de scikit-learn del paso 4.')


## 8. Ejercicios
1. Cambie las neuronas a (4,) y a (128,64). ¿Qué pasa con el MAE?
2. Quite `temp` de las variables. ¿Cuánto empeora?
3. ¿La red supera a la regresión lineal? ¿Valió la pena la complejidad?
4. Reentrene con `random_state` distinto y observe si el MAE cambia.